<a href="https://colab.research.google.com/github/madelsu/MOSAIC-Agentic-Severity-Phenotyping/blob/main/Phase_2_Patient_Classification/CLOSED_WEIGHT_FINAL_SET_UP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 📦 CELL 1: Unified Dependency Installation
# ============================================================

# We combine everything into one command so pip can find a compatible version of openai.
# 'crewai[tools]' already includes crewai, so we don't need to list both.
!pip install -U "crewai[tools]" litellm anthropic duckduckgo-search openpyxl tavily-python -q

# IMPORTANT: Force a specific OpenAI version that balances the two if conflicts persist
!pip install "openai>=1.83.0,<2.0.0" -q

In [ ]:
# ============================================================
# 🔑 CELL 2 UPDATED: API Keys + Model Config
# ============================================================

import os
from openai import OpenAI
from anthropic import Anthropic

os.environ["OPENAI_API_KEY"] = ""
os.environ["ANTHROPIC_API_KEY"] = ""
os.environ["DEEPSEEK_API_KEY"] = ""


# ── Model config ──────────────────────────────────────────────
# We now use GPT-4o for one of the assessors as requested.

ASSESSOR_1_MODEL   = "deepseek-chat"
ASSESSOR_2_MODEL   = "gpt-4o"
CONSOLIDATOR_MODEL = "claude-opus-4-6"
EXTRACTOR_MODEL    = "claude-sonnet-4-6"

ASSESSOR_1_LABEL   = "DeepSeek-Assessor"
ASSESSOR_2_LABEL   = "GPT4o-Assessor"
CONSOLIDATOR_LABEL = "Claude-Consolidator"
EXTRACTOR_LABEL    = "Claude-Extractor"

print(f"✅ Configuration Loaded:")
print(f" 📌 Assessor 1: {ASSESSOR_1_MODEL}")
print(f" 📌 Assessor 2: {ASSESSOR_2_MODEL}")

# ── Test OpenAI (New) ──────────────────────────────────────────
print("\n--- 🧪 Testing OpenAI (Assessor 2) ---")
try:
    # Uses default base_url for OpenAI
    client_openai = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    r_oa = client_openai.chat.completions.create(
        model=ASSESSOR_2_MODEL,
        max_tokens=10,
        messages=[{"role": "user", "content": "Hi"}]
    )
    print(f"✅ OpenAI works! → {r_oa.choices[0].message.content}")
except Exception as e:
    print(f"❌ OpenAI failed: {e}")

# ── Test DeepSeek ─────────────────────────────────────────────
print("\n--- 🧪 Testing DeepSeek (Assessor 1) ---")
try:
    # Explicitly set DeepSeek base_url
    client_deepseek = OpenAI(
        api_key=os.environ["DEEPSEEK_API_KEY"],
        base_url="https://api.deepseek.com"
    )
    r_ds = client_deepseek.chat.completions.create(
        model=ASSESSOR_1_MODEL,
        max_tokens=10,
        messages=[{"role": "user", "content": "Hi"}]
    )
    print(f"✅ DeepSeek works! → {r_ds.choices[0].message.content}")
except Exception as e:
    print(f"❌ DeepSeek failed: {e}")

# ── Test Claude ───────────────────────────────────────────────
print("\n--- 🧪 Testing Claude (Consolidator) ---")
try:
    client_claude = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    r_c = client_claude.messages.create(
        model=CONSOLIDATOR_MODEL,
        max_tokens=10,
        messages=[{"role": "user", "content": "Hi"}]
    )
    print(f"✅ Claude works! → {r_c.content[0].text}")
except Exception as e:
    print(f"❌ Claude failed: {e}")

print("\n" + "=" * 50)
print("🚀 All APIs tested and routing confirmed!")
print("=" * 50)

In [ ]:
# ============================================================
# RECOVERY CELL (LIGHT) — T5 Reclassification · N=200
#
# Use this if compressed_records_cache is ALREADY in memory
# (i.e. Cell A ran successfully in this session).
#
# Reloads:
#   1. Frozen framework (consolidated_framework_v3_T5.txt)
#   2. Checkpoint zip → completed_patients
#
# Skips: compression map (not needed — all 200 fit raw),
#        Excel file, cache rebuild
#
# After this cell: run Cell B directly.
# ============================================================

import os, json, zipfile
from google.colab import files as colab_files

# ── CONFIG ────────────────────────────────────────────────────
INDEX_DATE_LABEL   = "T5_reclassification"
INDEX_DATE_DISPLAY = "T5 — 5-Year Reclassification (reclass_date_5)"

OUTPUT_DIR   = "/content/pipeline_outputs"
INDEX_SUBDIR = f"{OUTPUT_DIR}/phase2/{INDEX_DATE_LABEL}"
os.makedirs(INDEX_SUBDIR, exist_ok=True)

print(f"{'=' * 60}")
print(f"LIGHT RECOVERY — {INDEX_DATE_DISPLAY}")
print(f"{'=' * 60}")


# ══════════════════════════════════════════════════════════════
# STEP 1: Check cache is in memory
# ══════════════════════════════════════════════════════════════
print(f"\nSTEP 1: Checking memory...")
try:
    cache_size = len(compressed_records_cache)
    ids_size   = len(ALL_PATIENT_IDS)
    print(f"  compressed_records_cache : {cache_size} patients")
    print(f"  ALL_PATIENT_IDS          : {ids_size} patients")
    if cache_size == 0 or ids_size == 0:
        print(f"  Cache is empty — run Cell A first, then re-run this cell")
except NameError:
    print(f"  compressed_records_cache not in memory")
    print(f"  Run Cell A first to rebuild it, then re-run this recovery cell")
    raise


# ══════════════════════════════════════════════════════════════
# STEP 2: Upload frozen framework v3
# ══════════════════════════════════════════════════════════════
print(f"\nSTEP 2: Upload consolidated_framework_v3_T5.txt")
uploaded = colab_files.upload()
fw_filename = list(uploaded.keys())[0]
with open(fw_filename, "r") as f:
    CONSOLIDATED_FRAMEWORK = f.read()
print(f"  Framework loaded ({len(CONSOLIDATED_FRAMEWORK):,} chars)")

# Sanity check — confirm new tier labels are present
expected_labels = ["Baseline T2D", "Mild Complications",
                   "Moderate Complications", "Advanced/Critical"]
missing = [l for l in expected_labels if l not in CONSOLIDATED_FRAMEWORK]
if missing:
    print(f"  WARNING: Expected tier labels missing from framework: {missing}")
    print(f"  Make sure you uploaded consolidated_framework_v3_T5.txt (not v2)")
else:
    print(f"  Tier labels confirmed: {expected_labels}")


# ══════════════════════════════════════════════════════════════
# STEP 3: Upload checkpoint zip → restore completed_patients
# ══════════════════════════════════════════════════════════════
print(f"\nSTEP 3: Upload your latest checkpoint zip...")
uploaded_cp  = colab_files.upload()
zip_filename = list(uploaded_cp.keys())[0]

with zipfile.ZipFile(zip_filename, "r") as zf:
    zf.extractall(OUTPUT_DIR)
print(f"  Checkpoint extracted to {OUTPUT_DIR}")

# Restore from checkpoint.json
checkpoint_path = f"{INDEX_SUBDIR}/checkpoint.json"
if os.path.exists(checkpoint_path):
    with open(checkpoint_path) as f:
        cp = json.load(f)
    completed_patients = set(cp["completed_patients"])
    print(f"  {len(completed_patients)} completed patients restored from checkpoint.json")
else:
    # Fallback: scan consolidated folder for saved classification files
    completed_patients = set()
    consol_dir = f"{INDEX_SUBDIR}/consolidated"
    if os.path.exists(consol_dir):
        for fname in os.listdir(consol_dir):
            if fname.startswith("classification_") and fname.endswith(".txt"):
                short = fname.replace("classification_", "").replace(".txt", "")
                for pid in ALL_PATIENT_IDS:
                    if pid[:8] == short:
                        completed_patients.add(pid)
                        break
    print(f"  {len(completed_patients)} completed patients recovered from files")


# ══════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════
remaining = [p for p in ALL_PATIENT_IDS if p not in completed_patients]

print(f"\n{'=' * 60}")
print(f"LIGHT RECOVERY COMPLETE")
print(f"{'=' * 60}")
print(f"  CONSOLIDATED_FRAMEWORK   : {len(CONSOLIDATED_FRAMEWORK):,} chars")
print(f"  ALL_PATIENT_IDS          : {len(ALL_PATIENT_IDS)} patients")
print(f"  compressed_records_cache : {len(compressed_records_cache)} patients")
print(f"  completed_patients       : {len(completed_patients)} done")
print(f"  Remaining                : {len(remaining)} to go")
print(f"{'=' * 60}")
print(f"\nRun Cell B now — it will resume from patient {len(completed_patients)+1}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL A — SIZE CHECK + COMPRESSION AUDIT  (T5 · N=200)
#
# WHAT THIS DOES:
#   1. Loads 200_patients_LLM_final_version_T5.xlsx
#   2. Measures raw record size for every patient (no compression yet)
#   3. Tells you exactly how many patients need compression vs can go raw
#   4. If compression IS needed: uploads MASTER_COMPRESSION_MAP and
#      builds smart_compress_patient() adapted for T5
#   5. Leaves compressed_records_cache in memory for Cell B (classification)
#
# KEY DIFFERENCES FROM OLD PIPELINE:
#   - Index date = reclass_date_5 (T5 = 5 years after first treatment)
#   - Data is already window-sliced in the Excel (no need to re-filter by date)
#   - Column PATIENT used throughout (not Id)
#   - No devices / imaging_studies sheets in this Excel
#   - Ground truth columns (GT_T5_DCSI_TIER etc.) are NOT present — clean file
#
# PREREQUISITES:
#   - 200_patients_LLM_final_version_T5.xlsx uploaded or on Drive
#   - CONSOLIDATED_FRAMEWORK in memory (or will estimate budget without it)
#   - master_compression_map.json (only needed if size check says compression required)
# ══════════════════════════════════════════════════════════════════════════════

import os, re, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files as colab_files

# ─────────────────────────────────────────────────────────────────────────────
# 0. CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
INDEX_DATE_LABEL   = "T5_reclassification"   # label used in patient record headers
MAX_OBS_HISTORY    = 5                        # max readings per obs type (compression only)

# Context budget — bottleneck is DeepSeek at 64K tokens
FRAMEWORK_CHARS    = len(CONSOLIDATED_FRAMEWORK) if "CONSOLIDATED_FRAMEWORK" in dir() else 8_000
PROMPT_OVERHEAD    = 3_000
OVERHEAD_TOKENS    = (FRAMEWORK_CHARS + PROMPT_OVERHEAD) // 4
BOTTLENECK_LIMIT   = 64_000
AVAILABLE          = BOTTLENECK_LIMIT - OVERHEAD_TOKENS
SAFE_LIMIT_TOKENS  = int(AVAILABLE * 0.80)
SAFE_LIMIT_CHARS   = SAFE_LIMIT_TOKENS * 4
HARD_MAX_CHARS     = AVAILABLE * 4

print("=" * 65)
print("CELL A — SIZE CHECK + COMPRESSION AUDIT (T5 · N=200)")
print("=" * 65)
print(f"\n  Context budget:")
print(f"    Framework + overhead : ~{OVERHEAD_TOKENS:,} tokens")
print(f"    Safe limit (80%)     : ~{SAFE_LIMIT_TOKENS:,} tokens  /  {SAFE_LIMIT_CHARS:,} chars")
print(f"    Hard ceiling         : ~{AVAILABLE:,} tokens  /  {HARD_MAX_CHARS:,} chars\n")


# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — UPLOAD AND LOAD EXCEL
# ─────────────────────────────────────────────────────────────────────────────
print("STEP 1 — Upload 200_patients_LLM_final_version_T5.xlsx")
uploaded_xl   = colab_files.upload()
PIPELINE_FILE = list(uploaded_xl.keys())[0]
print(f"  Loaded: {PIPELINE_FILE}")

xl            = pd.ExcelFile(PIPELINE_FILE)
print(f"  Sheets found: {xl.sheet_names}\n")

# Load all sheets — gracefully skip any that don't exist
def load_sheet(name):
    if name in xl.sheet_names:
        return pd.read_excel(PIPELINE_FILE, sheet_name=name)
    return pd.DataFrame()

patient_list  = load_sheet("patient_list")
patients      = load_sheet("patients")
conditions    = load_sheet("conditions")
observations  = load_sheet("observations")
medications   = load_sheet("medications")
encounters    = load_sheet("encounters")
careplans     = load_sheet("careplans")
allergies     = load_sheet("allergies")
procedures    = load_sheet("procedures")
immunizations = load_sheet("immunizations")

ALL_PATIENT_IDS = patient_list["PATIENT"].tolist()
print(f"  Patients loaded : {len(ALL_PATIENT_IDS)}")

# ── Column sanity check ────────────────────────────────────────
print("\n  Column check:")
for sheet_name, df in [
    ("patient_list",  patient_list),
    ("patients",      patients),
    ("conditions",    conditions),
    ("observations",  observations),
    ("medications",   medications),
]:
    pat_col = "PATIENT" if "PATIENT" in df.columns else ("Id" if "Id" in df.columns else "MISSING")
    n_rows  = len(df)
    print(f"    {sheet_name:<15}  {n_rows:>6,} rows   patient col = '{pat_col}'")

# ── Ground truth leak check ────────────────────────────────────
forbidden = ["GT_T5_DCSI", "T5_DCSI_CAT", "GROUND_TRUTH", "DIED_IN_FOLLOWUP",
             "DEATH_DATE", "DEATHDATE"]
leaked = [
    f"{s}.{c}"
    for s in xl.sheet_names
    for c in pd.read_excel(PIPELINE_FILE, sheet_name=s).columns
    if any(f.upper() in str(c).upper() for f in forbidden)
]
if leaked:
    print(f"\n  WARNING — potential ground truth columns detected:")
    for col in leaked:
        print(f"    {col}")
else:
    print(f"\n  No ground truth leakage detected")


# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — BUILD RAW RECORDS AND MEASURE SIZES
# Data is already window-sliced in the Excel, so no date filtering needed here.
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("STEP 2 — Measuring raw record sizes (no compression)")
print("=" * 65)

PII_COLS = {"SSN", "DRIVERS", "PASSPORT", "ADDRESS", "CITY", "STATE",
            "COUNTY", "ZIP", "LAT", "LON", "HEALTHCARE_EXPENSES",
            "HEALTHCARE_COVERAGE"}

def build_raw_record(patient_id):
    """
    Assemble a full plain-text patient record from all sheets.
    Data is already covariate-window-sliced — no date filtering needed.
    PII columns are stripped; all clinical rows are included as-is.
    """
    record = (
        f"=== PATIENT RECORD: {patient_id} ===\n"
        f"=== INDEX DATE: {INDEX_DATE_LABEL.upper()} "
        f"(5 years after first T2D treatment) ===\n\n"
    )

    # Demographics
    demo = patients[patients.get("PATIENT", patients.get("Id", pd.Series())).eq(patient_id)]
    if "Id" in patients.columns:
        demo = patients[patients["Id"] == patient_id]
    elif "PATIENT" in patients.columns:
        demo = patients[patients["PATIENT"] == patient_id]
    else:
        demo = pd.DataFrame()

    if not demo.empty:
        keep = [c for c in demo.columns if c not in PII_COLS]
        record += f"--- DEMOGRAPHICS ---\n{demo[keep].to_string(index=False)}\n\n"
    else:
        record += "--- DEMOGRAPHICS ---\nNo demographics found.\n\n"

    # Index date metadata from patient_list
    meta = patient_list[patient_list["PATIENT"] == patient_id]
    if not meta.empty:
        record += f"--- INDEX DATE METADATA ---\n{meta.to_string(index=False)}\n\n"

    # Clinical sheets
    clinical = [
        ("CONDITIONS",    conditions,    ["PATIENT", "ENCOUNTER"]),
        ("OBSERVATIONS",  observations,  ["PATIENT", "ENCOUNTER", "TYPE"]),
        ("MEDICATIONS",   medications,   ["PATIENT", "ENCOUNTER", "PAYER"]),
        ("ENCOUNTERS",    encounters,    ["PATIENT"]),
        ("PROCEDURES",    procedures,    ["PATIENT", "ENCOUNTER"]),
        ("CAREPLANS",     careplans,     ["PATIENT", "ENCOUNTER"]),
        ("ALLERGIES",     allergies,     ["PATIENT", "ENCOUNTER"]),
        ("IMMUNIZATIONS", immunizations, ["PATIENT", "ENCOUNTER"]),
    ]

    for section_name, df, drop_cols in clinical:
        if df.empty:
            record += f"--- {section_name} ---\nSheet not present in Excel.\n\n"
            continue
        pat_df = df[df["PATIENT"] == patient_id]
        if not pat_df.empty:
            keep = [c for c in pat_df.columns if c not in drop_cols]
            record += (
                f"--- {section_name} ({len(pat_df)} rows) ---\n"
                f"{pat_df[keep].to_string(index=False)}\n\n"
            )
        else:
            record += f"--- {section_name} ---\nNo records in covariate window.\n\n"

    return record


# Measure all 200 patients
print(f"\n  Building raw records for {len(ALL_PATIENT_IDS)} patients...")
raw_sizes   = []
raw_records = {}

for i, pid in enumerate(ALL_PATIENT_IDS):
    try:
        record = build_raw_record(pid)
        raw_records[pid] = record
        chars  = len(record)
        tokens = chars // 4

        # Sparsity check
        has_conditions   = conditions[conditions["PATIENT"] == pid].shape[0] > 0  if not conditions.empty   else False
        has_observations = observations[observations["PATIENT"] == pid].shape[0] > 0 if not observations.empty else False
        has_medications  = medications[medications["PATIENT"] == pid].shape[0] > 0  if not medications.empty  else False
        n_fields         = sum([has_conditions, has_observations, has_medications])

        raw_sizes.append({
            "patient_id":         pid,
            "raw_chars":          chars,
            "estimated_tokens":   tokens,
            "fits_safely":        chars <= SAFE_LIMIT_CHARS,
            "fits_at_all":        chars <= HARD_MAX_CHARS,
            "needs_compression":  chars >  SAFE_LIMIT_CHARS,
            "data_fields":        n_fields,
            "sparse":             n_fields == 0,
        })
    except Exception as e:
        print(f"  ERROR on {pid[:8]}: {e}")

    if (i + 1) % 50 == 0:
        print(f"  ...{i+1}/{len(ALL_PATIENT_IDS)} done")

sizes_df = pd.DataFrame(raw_sizes)
valid    = sizes_df[sizes_df["raw_chars"] > 0]


# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — SIZE REPORT
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("SIZE REPORT")
print("=" * 65)

n_safe    = valid["fits_safely"].sum()
n_over    = (~valid["fits_safely"] & valid["fits_at_all"]).sum()
n_wontfit = (~valid["fits_at_all"]).sum()
n_sparse  = valid["sparse"].sum()

print(f"\n  Total patients        : {len(valid)}")
print(f"\n  Raw size (chars):")
print(f"    Min    : {valid['raw_chars'].min():>10,}")
print(f"    Median : {valid['raw_chars'].median():>10,.0f}")
print(f"    Mean   : {valid['raw_chars'].mean():>10,.0f}")
print(f"    Max    : {valid['raw_chars'].max():>10,}")

print(f"\n  Context window fit (RAW, no compression):")
print(f"    Fits safely (<= {SAFE_LIMIT_CHARS:,} chars)  : {n_safe:>3} / {len(valid)}")
print(f"    Over safe limit, still fits        : {n_over:>3} / {len(valid)}")
print(f"    Won't fit at all (> {HARD_MAX_CHARS:,} chars) : {n_wontfit:>3} / {len(valid)}")

print(f"\n  Data sparsity:")
for n, label in [(3, "All 3 fields (conditions + obs + meds)"),
                 (2, "2 fields present"),
                 (1, "1 field only"),
                 (0, "All empty — sparse record")]:
    count = (valid["data_fields"] == n).sum()
    bar   = "█" * min(count // 2, 30)
    print(f"    {label:<42}: {count:>3}  {bar}")

print(f"\n  Size distribution:")
bins = [0, 5_000, 10_000, 20_000, 40_000, 80_000, 160_000, 500_000]
for lo, hi in zip(bins, bins[1:]):
    count = len(valid[(valid["raw_chars"] >= lo) & (valid["raw_chars"] < hi)])
    if count > 0:
        bar = "█" * min(count, 40)
        print(f"    {lo:>7,}–{hi:>7,} chars : {count:>3}  {bar}")

# Size distribution plot
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(valid["raw_chars"], bins=30, color="#4472C4", edgecolor="white", alpha=0.85)
ax.axvline(SAFE_LIMIT_CHARS, color="#D62728", linewidth=2, linestyle="--",
           label=f"Safe limit ({SAFE_LIMIT_CHARS:,} chars)")
ax.axvline(HARD_MAX_CHARS,   color="#FF7F0E", linewidth=2, linestyle=":",
           label=f"Hard max ({HARD_MAX_CHARS:,} chars)")
ax.set_xlabel("Raw record size (chars)", fontsize=11)
ax.set_ylabel("Number of patients", fontsize=11)
ax.set_title("T5 Cohort — Raw Record Size Distribution (N=200)", fontsize=12, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 — VERDICT + DECIDE COMPRESSION STRATEGY
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("VERDICT")
print("=" * 65)

NEEDS_COMPRESSION = n_over > 0 or n_wontfit > 0

if not NEEDS_COMPRESSION:
    print(f"\n  ALL {len(valid)} patients fit within the safe context limit.")
    print(f"  COMPRESSION NOT NEEDED.")
    print(f"\n  Strategy: pass raw records directly to LLM agents.")
    print(f"  compressed_records_cache = raw_records (all uncompressed)")
    compressed_records_cache = raw_records
    print(f"\n  GO — ready for Cell B")

else:
    print(f"\n  {n_over + n_wontfit} patients exceed the safe limit.")
    print(f"  COMPRESSION REQUIRED for those patients.")
    print(f"  Strategy: raw for patients that fit, compressed for the rest.")
    print(f"\n  --> Proceeding to STEP 5: upload MASTER_COMPRESSION_MAP")

    # ── STEP 5: Upload compression map ────────────────────────
    print("\n" + "=" * 65)
    print("STEP 5 — Upload MASTER_COMPRESSION_MAP")
    print("=" * 65)
    print("\n  Upload master_compression_map.json (or .txt)")
    uploaded_map = colab_files.upload()
    map_filename = list(uploaded_map.keys())[0]

    with open(map_filename, "r") as f:
        raw_content = f.read().strip()

    try:
        parsed = json.loads(raw_content)
    except json.JSONDecodeError:
        json_match = re.search(r'\{.*\}', raw_content, re.DOTALL)
        parsed = json.loads(json_match.group()) if json_match else {}

    # Unwrap if nested
    if "master_compression_map" in parsed:
        MASTER_COMPRESSION_MAP = parsed["master_compression_map"]
    elif "observations" in parsed and "medications" in parsed:
        MASTER_COMPRESSION_MAP = parsed
    else:
        first_val = list(parsed.values())[0]
        MASTER_COMPRESSION_MAP = first_val if isinstance(first_val, dict) else parsed

    total_descs = sum(len(v) for v in MASTER_COMPRESSION_MAP.values())
    print(f"  Loaded: {total_descs} descriptions across {len(MASTER_COMPRESSION_MAP)} tables")
    for table, descs in MASTER_COMPRESSION_MAP.items():
        print(f"    {table:<15}: {len(descs)} strings")

    # ── STEP 6: Build compressed version ──────────────────────
    print("\n" + "=" * 65)
    print("STEP 6 — Building smart_compress_patient() for T5")
    print("=" * 65)

    def get_desc_col(df):
        for col in ["DESCRIPTION", "REASONDESCRIPTION"]:
            if col in df.columns:
                return col
        raise KeyError(f"No description column found. Columns: {list(df.columns)}")

    def build_compressed_record(patient_id):
        """
        Apply framework filter + obs history cap.
        Used only when the raw record exceeds SAFE_LIMIT_CHARS.
        Data is already window-sliced — no date filtering needed.
        """
        record = (
            f"=== PATIENT RECORD: {patient_id} ===\n"
            f"=== INDEX DATE: {INDEX_DATE_LABEL.upper()} "
            f"(5 years after first T2D treatment) ===\n\n"
        )

        # Demographics — always pass through
        if "Id" in patients.columns:
            demo = patients[patients["Id"] == patient_id]
        else:
            demo = patients[patients["PATIENT"] == patient_id]
        if not demo.empty:
            keep = [c for c in demo.columns if c not in PII_COLS]
            record += f"--- DEMOGRAPHICS ---\n{demo[keep].to_string(index=False)}\n\n"
        else:
            record += "--- DEMOGRAPHICS ---\nNo demographics found.\n\n"

        meta = patient_list[patient_list["PATIENT"] == patient_id]
        if not meta.empty:
            record += f"--- INDEX DATE METADATA ---\n{meta.to_string(index=False)}\n\n"

        # Conditions — framework filter
        cond = conditions[conditions["PATIENT"] == patient_id] if not conditions.empty else pd.DataFrame()
        if not cond.empty:
            desc_col   = get_desc_col(cond)
            keep_descs = set(MASTER_COMPRESSION_MAP.get("conditions", []))
            relevant   = cond[cond[desc_col].isin(keep_descs)]
            drop_cols  = ["PATIENT", "ENCOUNTER"]
            keep_cols  = [c for c in cond.columns if c not in drop_cols]
            source     = relevant if not relevant.empty else cond
            note       = (f"{len(cond)} total, {len(relevant)} framework-relevant"
                          if not relevant.empty
                          else f"{len(cond)} total, 0 matched framework — showing all")
            record += f"--- CONDITIONS [{note}] ---\n{source[keep_cols].to_string(index=False)}\n\n"
        else:
            record += "--- CONDITIONS ---\nNo records in covariate window.\n\n"

        # Observations — framework filter + last N per type
        obs = observations[observations["PATIENT"] == patient_id] if not observations.empty else pd.DataFrame()
        if not obs.empty:
            desc_col   = get_desc_col(obs)
            keep_descs = set(MASTER_COMPRESSION_MAP.get("observations", []))
            relevant   = obs[obs[desc_col].isin(keep_descs)].copy()
            drop_cols  = ["PATIENT", "ENCOUNTER", "TYPE"]
            keep_cols  = [c for c in obs.columns if c not in drop_cols]
            if not relevant.empty:
                relevant["DATE"] = pd.to_datetime(relevant["DATE"], errors="coerce")
                relevant = relevant.sort_values("DATE", ascending=False)
                compressed = relevant.groupby(desc_col).head(MAX_OBS_HISTORY)
                compressed = compressed.sort_values([desc_col, "DATE"], ascending=[True, False])
                note = (f"{len(obs)} total → {len(relevant)} framework-relevant "
                        f"→ {len(compressed)} after last {MAX_OBS_HISTORY} per type")
                source = compressed
            else:
                note   = f"{len(obs)} total, 0 matched framework — showing all"
                source = obs
            src_cols = [c for c in keep_cols if c in source.columns]
            record += f"--- OBSERVATIONS [{note}] ---\n{source[src_cols].to_string(index=False)}\n\n"
        else:
            record += "--- OBSERVATIONS ---\nNo records in covariate window.\n\n"

        # Medications — framework filter
        med = medications[medications["PATIENT"] == patient_id] if not medications.empty else pd.DataFrame()
        if not med.empty:
            desc_col   = get_desc_col(med)
            keep_descs = set(MASTER_COMPRESSION_MAP.get("medications", []))
            relevant   = med[med[desc_col].isin(keep_descs)]
            drop_cols  = ["PATIENT", "ENCOUNTER", "PAYER"]
            keep_cols  = [c for c in med.columns if c not in drop_cols]
            source     = relevant if not relevant.empty else med
            note       = (f"{len(med)} total, {len(relevant)} framework-relevant"
                          if not relevant.empty
                          else f"{len(med)} total, 0 matched — showing all")
            record += f"--- MEDICATIONS [{note}] ---\n{source[keep_cols].to_string(index=False)}\n\n"
        else:
            record += "--- MEDICATIONS ---\nNo records in covariate window.\n\n"

        # Procedures — framework filter
        proc = procedures[procedures["PATIENT"] == patient_id] if not procedures.empty else pd.DataFrame()
        if not proc.empty:
            desc_col   = get_desc_col(proc)
            keep_descs = set(MASTER_COMPRESSION_MAP.get("procedures", []))
            relevant   = proc[proc[desc_col].isin(keep_descs)]
            drop_cols  = ["PATIENT", "ENCOUNTER"]
            keep_cols  = [c for c in proc.columns if c not in drop_cols]
            source     = relevant if not relevant.empty else proc
            note       = (f"{len(proc)} total, {len(relevant)} framework-relevant"
                          if not relevant.empty
                          else f"{len(proc)} total, 0 matched — showing all")
            record += f"--- PROCEDURES [{note}] ---\n{source[keep_cols].to_string(index=False)}\n\n"
        else:
            record += "--- PROCEDURES ---\nNo records in covariate window.\n\n"

        # Careplans — framework filter
        cp = careplans[careplans["PATIENT"] == patient_id] if not careplans.empty else pd.DataFrame()
        if not cp.empty:
            desc_col   = get_desc_col(cp)
            keep_descs = set(MASTER_COMPRESSION_MAP.get("careplans", []))
            relevant   = cp[cp[desc_col].isin(keep_descs)]
            drop_cols  = ["PATIENT", "ENCOUNTER"]
            keep_cols  = [c for c in cp.columns if c not in drop_cols]
            source     = relevant if not relevant.empty else cp
            note       = (f"{len(cp)} total, {len(relevant)} framework-relevant"
                          if not relevant.empty
                          else f"{len(cp)} total, 0 matched — showing all")
            record += f"--- CAREPLANS [{note}] ---\n{source[keep_cols].to_string(index=False)}\n\n"
        else:
            record += "--- CAREPLANS ---\nNo records in covariate window.\n\n"

        # Allergies + immunizations — always pass through (small)
        for section_name, df, drop_cols in [
            ("ALLERGIES",     allergies,     ["PATIENT", "ENCOUNTER"]),
            ("IMMUNIZATIONS", immunizations, ["PATIENT", "ENCOUNTER"]),
            ("ENCOUNTERS",    encounters,    ["PATIENT"]),
        ]:
            if df.empty:
                record += f"--- {section_name} ---\nSheet not present.\n\n"
                continue
            pat_df = df[df["PATIENT"] == patient_id]
            if not pat_df.empty:
                keep = [c for c in pat_df.columns if c not in drop_cols]
                record += f"--- {section_name} ({len(pat_df)} rows) ---\n{pat_df[keep].to_string(index=False)}\n\n"
            else:
                record += f"--- {section_name} ---\nNo records in covariate window.\n\n"

        return record

    def smart_compress_patient(patient_id):
        """Use raw record if it fits; apply compression only if it doesn't."""
        raw = raw_records.get(patient_id) or build_raw_record(patient_id)
        if len(raw) <= SAFE_LIMIT_CHARS:
            return raw, "raw"
        compressed = build_compressed_record(patient_id)
        return compressed, "compressed"

    print("  smart_compress_patient() built\n")

    # ── STEP 7: Build compressed_records_cache ────────────────
    print("STEP 7 — Building compressed_records_cache for all 200 patients")

    compressed_records_cache = {}
    mode_counts = {"raw": 0, "compressed": 0}

    oversized_ids = set(
        valid[valid["needs_compression"]]["patient_id"].tolist()
    )

    for i, pid in enumerate(ALL_PATIENT_IDS):
        if pid in oversized_ids:
            record, mode = smart_compress_patient(pid)
        else:
            record, mode = raw_records[pid], "raw"
        compressed_records_cache[pid] = record
        mode_counts[mode] += 1
        if (i + 1) % 50 == 0:
            print(f"  ...{i+1}/{len(ALL_PATIENT_IDS)} done")

    print(f"\n  Cache built:")
    print(f"    Raw        : {mode_counts['raw']:>3} patients")
    print(f"    Compressed : {mode_counts['compressed']:>3} patients")

    # Re-measure after compression
    final_sizes = {pid: len(rec) for pid, rec in compressed_records_cache.items()}
    still_over  = [pid for pid, sz in final_sizes.items() if sz > HARD_MAX_CHARS]
    if still_over:
        print(f"\n  WARNING: {len(still_over)} patients STILL over hard limit after compression.")
        print(f"  Consider reducing MAX_OBS_HISTORY from {MAX_OBS_HISTORY} to 3 and re-running.")
        for pid in still_over:
            print(f"    {pid[:8]}  {final_sizes[pid]:,} chars")
    else:
        print(f"\n  All patients within hard limit after compression.")

    print(f"\n  GO — ready for Cell B")


# ─────────────────────────────────────────────────────────────────────────────
# FINAL SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("SUMMARY")
print("=" * 65)
print(f"  compressed_records_cache : {len(compressed_records_cache)} patients")
print(f"  Index date label         : {INDEX_DATE_LABEL}")
print(f"  Compression used         : {NEEDS_COMPRESSION}")
print(f"\n  Do NOT restart the runtime before running Cell B.")
print(f"  compressed_records_cache must stay in memory.")

In [ ]:
# ── Upload Frozen Framework ───────────────────────────────────
from google.colab import files as colab_files

print("📤 Upload your consolidated_framework_v2.txt file...")
uploaded = colab_files.upload()
filename = list(uploaded.keys())[0]

with open(filename, "r") as f:
    CONSOLIDATED_FRAMEWORK = f.read()

print(f"✅ CONSOLIDATED_FRAMEWORK loaded ({len(CONSOLIDATED_FRAMEWORK):,} chars)")
print(f"   Preview: {CONSOLIDATED_FRAMEWORK[:200]}...")

In [ ]:
# ============================================================
# CELL B: Phase 2 Pipeline — T5 Reclassification · N=200
#
# ARCHITECTURE:
#   Step 1: Pull raw record from cache (built by Cell A)
#   Step 2: GPT-4o (Dr. A) + DeepSeek (Dr. B) assess ASYNC in parallel
#   Step 3: Claude Sonnet consolidates → final classification
#
# KEY DIFFERENCES FROM ORIGINAL PIPELINE:
#   - Index date = T5 (5 years after first T2D treatment)
#   - Tier labels: Baseline T2D / Mild Complications /
#                  Moderate Complications / Advanced/Critical
#   - NO "Insufficient Data" tier — sparse records = Baseline T2D
#     (all patients have confirmed T2D; absence of complications
#      IS clinically meaningful and maps to Baseline T2D)
#   - LLM instructed to use ALL available EHR data holistically,
#     not just the 8 framework domains
#   - Binary: Advanced/Critical + Moderate Complications = COMPLEX
#             Mild Complications + Baseline T2D = NOT_COMPLEX
#   - Score fields renamed to match new tier labels
#
# PREREQUISITES:
#   - compressed_records_cache built by Cell A (same runtime session)
#   - ALL_PATIENT_IDS defined by Cell A
#   - CONSOLIDATED_FRAMEWORK loaded (consolidated_framework_v3_T5.txt)
#   - API keys set in environment
# ============================================================

from crewai import Agent, Task, Crew
import time, json, os, zipfile, math
from google.colab import files as colab_files

# ── CONFIG ────────────────────────────────────────────────────
INDEX_DATE_LABEL = "T5_reclassification"
INDEX_DATE_DISPLAY = "T5 — 5-Year Reclassification (5 years after first T2D treatment)"

BATCH_SIZE       = 10   # patients per batch before pausing
CHECKPOINT_EVERY = 2    # download checkpoint every N patients

OUTPUT_DIR   = "/content/pipeline_outputs"
INDEX_SUBDIR = f"{OUTPUT_DIR}/phase2/{INDEX_DATE_LABEL}"
os.makedirs(INDEX_SUBDIR, exist_ok=True)

# ── Tier definitions (must match ground truth labels exactly) ─
VALID_TIERS = [
    "Baseline T2D",
    "Mild Complications",
    "Moderate Complications",
    "Advanced/Critical",
]

# Binary mapping
# COMPLEX   = Moderate Complications + Advanced/Critical  (established disease burden)
# NOT_COMPLEX = Baseline T2D + Mild Complications          (manageable / early)
COMPLEX_TIERS = {"Moderate Complications", "Advanced/Critical"}

# ── Utilities ─────────────────────────────────────────────────
def save_and_track(subpath, filename, content):
    path = f"{INDEX_SUBDIR}/{subpath}/{filename}"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if isinstance(content, (dict, list)):
        with open(path + ".json", "w") as f:
            json.dump(content, f, indent=2)
    with open(path + ".txt", "w") as f:
        f.write(json.dumps(content, indent=2)
                if isinstance(content, (dict, list)) else str(content))


def softmax_scores(s_baseline, s_mild, s_moderate, s_advanced):
    """Convert four 0-100 scores to softmax probability distribution."""
    scores = [s_baseline or 0, s_mild or 0, s_moderate or 0, s_advanced or 0]
    exps   = [math.exp(s) for s in scores]
    total  = sum(exps)
    probs  = [round(e / total, 4) for e in exps]
    return {
        "prob_baseline_t2d"          : probs[0],
        "prob_mild_complications"    : probs[1],
        "prob_moderate_complications": probs[2],
        "prob_advanced_critical"     : probs[3],
    }


def parse_score(parsed_dict, key):
    try:
        return int(float(parsed_dict.get(key, "").strip()))
    except (ValueError, TypeError, AttributeError):
        return None


def download_checkpoint_zip(completed_set, all_results, errors):
    checkpoint_meta = {
        "index_date"         : INDEX_DATE_LABEL,
        "completed_patients" : list(completed_set),
        "total_patients"     : len(ALL_PATIENT_IDS),
        "errors"             : len(errors),
        "timestamp"          : time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    with open(f"{INDEX_SUBDIR}/checkpoint.json", "w") as f:
        json.dump(checkpoint_meta, f, indent=2)
    save_and_track("metadata", "pipeline_results_partial", all_results)
    if errors:
        save_and_track("metadata", "pipeline_errors_partial", errors)

    zip_name = f"/content/pipeline_checkpoint_{INDEX_DATE_LABEL}.zip"
    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files_list in os.walk(INDEX_SUBDIR):
            for fname in files_list:
                fpath   = os.path.join(root, fname)
                arcname = os.path.relpath(fpath, OUTPUT_DIR)
                zf.write(fpath, arcname)
    size_mb = os.path.getsize(zip_name) / (1024 * 1024)
    print(f"  Checkpoint saved: {len(completed_set)} patients, {size_mb:.1f} MB")
    colab_files.download(zip_name)


def load_checkpoint():
    print(f"Upload checkpoint zip for {INDEX_DATE_LABEL}...")
    uploaded = colab_files.upload()
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name, "r") as zf:
        zf.extractall(OUTPUT_DIR)
    checkpoint_path = f"{INDEX_SUBDIR}/checkpoint.json"
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path) as f:
            cp = json.load(f)
        completed = set(cp["completed_patients"])
        print(f"Resumed: {len(completed)} done, "
              f"{len(ALL_PATIENT_IDS) - len(completed)} remaining")
        return completed
    # Fallback: scan consolidated folder
    completed = set()
    consol_dir = f"{INDEX_SUBDIR}/consolidated"
    if os.path.exists(consol_dir):
        for fname in os.listdir(consol_dir):
            if fname.startswith("classification_") and fname.endswith(".txt"):
                short = fname.replace("classification_", "").replace(".txt", "")
                for pid in ALL_PATIENT_IDS:
                    if pid[:8] == short:
                        completed.add(pid)
                        break
    print(f"Recovered {len(completed)} completed patients from files")
    return completed


# ── Resume check ──────────────────────────────────────────────
# To resume after a crash, uncomment:
# completed_patients = load_checkpoint()

if "completed_patients" not in dir() or not isinstance(completed_patients, set):
    completed_patients = set()

remaining_all = [pid for pid in ALL_PATIENT_IDS if pid not in completed_patients]
remaining     = remaining_all[:BATCH_SIZE]

print(f"{'=' * 65}")
print(f"PHASE 2 — {INDEX_DATE_DISPLAY}")
print(f"{'=' * 65}")
print(f"  Total patients  : {len(ALL_PATIENT_IDS)}")
print(f"  Already done    : {len(completed_patients)}")
print(f"  Remaining total : {len(remaining_all)}")
print(f"  This batch      : {len(remaining)}")
print(f"  Checkpoint every: {CHECKPOINT_EVERY} patients")
print(f"  Output dir      : {INDEX_SUBDIR}")
print(f"{'=' * 65}")

if not remaining:
    print("All patients already completed!")
else:

    # ══════════════════════════════════════════════════════════
    # AGENT DEFINITIONS
    # ══════════════════════════════════════════════════════════

    assessor_gpt = Agent(
        role="Dr. A — Clinical Informatician",
        goal=(
            "Classify this T2D patient's severity tier at their 5-year "
            "reclassification point using the provided framework AND all available "
            "clinical evidence in the record. Be methodical, data-driven, and "
            "evidence-based. Use everything in the record — not just lab values."
        ),
        backstory=(
            "You are a clinical informatician specialising in EHR-based T2D phenotyping. "
            "You are methodical and conservative — you only count what is explicitly "
            "documented. You systematically work through the framework domains but you "
            "also look at the broader clinical picture: medication trajectory, encounter "
            "frequency, comorbidity burden, and care plan activity. "
            "You never refuse to classify — if a patient has T2D with no documented "
            "complications, that is clinically meaningful and correctly classified as "
            "Baseline T2D, not 'insufficient data'."
        ),
        verbose=True,
        allow_delegation=False,
        llm="gpt-4o"
    )

    assessor_deepseek = Agent(
        role="Dr. B — Consultant Endocrinologist",
        goal=(
            "Classify this T2D patient's severity tier at their 5-year "
            "reclassification point using the provided framework AND all available "
            "clinical evidence. Take a holistic, outcomes-focused approach."
        ),
        backstory=(
            "You are a consultant endocrinologist with 20 years of clinical experience "
            "in T2D management. You take a holistic view — you consider the full clinical "
            "trajectory over the 5-year window: what conditions appeared, what medications "
            "were started or escalated, how labs evolved over time, what specialist referrals "
            "or care plans are present. "
            "You also read between the lines — a patient with frequent encounters and "
            "escalating medications tells a different story than one with a single annual "
            "review, even if the lab values are similar. "
            "You never use 'Insufficient Data' as a classification — a patient with "
            "confirmed T2D and no complication evidence is correctly classified as "
            "Baseline T2D. Absence of complications IS a finding."
        ),
        verbose=True,
        allow_delegation=False,
        llm="deepseek/deepseek-chat"
    )

    consolidator = Agent(
        role="Chief Medical Informatician — Final Arbiter",
        goal=(
            "Review both assessors' classifications, resolve disagreements, and "
            "produce a final severity classification with full justification."
        ),
        backstory=(
            "You are the chief medical informatician responsible for the final "
            "phenotyping decision at the T5 reclassification point. "
            "You review both assessors' reasoning, identify agreement and disagreement, "
            "and produce a definitive classification. "
            "When assessors disagree, you examine the specific evidence each cited "
            "and determine which interpretation is better supported. "
            "You never classify as 'Insufficient Data' — all patients have confirmed "
            "T2D and a 5-year EHR window; the correct classification for a patient "
            "with no documented complications is Baseline T2D."
        ),
        verbose=True,
        allow_delegation=False,
        llm="anthropic/claude-sonnet-4-6"
    )

    # ══════════════════════════════════════════════════════════
    # MAIN LOOP
    # ══════════════════════════════════════════════════════════
    all_results               = []
    errors                    = []
    patients_since_checkpoint = 0
    pipeline_start            = time.time()

    for idx, patient_id in enumerate(remaining):
        patient_short     = patient_id[:8]
        raw_record        = compressed_records_cache[patient_id]
        overall_idx       = len(completed_patients) + 1

        print(f"\n{'=' * 65}")
        print(f"Patient {overall_idx}/{len(ALL_PATIENT_IDS)}: {patient_short} "
              f"({len(raw_record):,} chars, ~{len(raw_record)//4:,} tokens)")
        print(f"{'=' * 65}")

        # ── Assessment prompt (shared by both assessors) ──────
        ASSESS_PROMPT = f"""
You are classifying a Type 2 Diabetes patient's severity at their
T5 reclassification point (5 years after first T2D treatment).

The clinical data in the record covers the full window from first treatment
up to and including the T5 index date. Use EVERYTHING in the record.

=== CLASSIFICATION TIERS (use these exact labels) ===
  Tier 1: Baseline T2D
  Tier 2: Mild Complications
  Tier 3: Moderate Complications
  Tier 4: Advanced/Critical

IMPORTANT — SPARSE RECORDS:
All patients in this cohort have CONFIRMED Type 2 Diabetes.
If the record contains little clinical data beyond the T2D diagnosis:
  → DO NOT classify as "Insufficient Data"
  → Absence of documented complications IS a valid clinical finding
  → Classify as "Baseline T2D" and explain that no complication evidence
    was found in the 5-year window

=== CONSOLIDATED SEVERITY PHENOTYPING FRAMEWORK ===
{CONSOLIDATED_FRAMEWORK}
=== END FRAMEWORK ===

=== PATIENT CLINICAL RECORD (T5 covariate window) ===
{raw_record}
=== END RECORD ===

INSTRUCTIONS — USE ALL AVAILABLE EVIDENCE:

1. FRAMEWORK DOMAINS (systematic):
   Work through each domain in the framework. For each, state:
   - What evidence is present in the record
   - What threshold it meets (or does not meet)
   - Which tier it triggers

2. BEYOND THE FRAMEWORK — also consider:
   - Medication trajectory: How many antidiabetic agents? Any insulin?
     Escalation over time suggests worsening control.
   - Encounter frequency and type: Frequent ED visits, hospitalisations,
     or specialist referrals indicate higher burden.
   - Comorbidity pattern: Hypertension + dyslipidemia together vs alone.
   - Care plan content: Active diabetes management plans, foot care,
     ophthalmology referrals.
   - Lab trends over time: Worsening eGFR, rising HbA1c, new proteinuria.
   - Condition onset dates: Did complications develop early or late in window?

3. FINAL TIER ASSIGNMENT:
   Apply Framework Rule 0 (highest tier triggered across any domain).
   Apply Framework Rule 0a (Advanced/Critical if ≥2 Moderate Complications
   domain criteria are met simultaneously).

REQUIRED OUTPUT FORMAT:
- Domain-by-domain assessment with specific evidence citations
- Beyond-framework clinical observations
- Summary: which criteria ARE met vs NOT met
- Final Four-Tier Classification: [Baseline T2D / Mild Complications /
  Moderate Complications / Advanced/Critical]
- Final Binary Classification:
    COMPLEX     = Moderate Complications OR Advanced/Critical
    NOT_COMPLEX = Baseline T2D OR Mild Complications
- Confidence: [High / Medium / Low]
- Key uncertainties or data gaps

TIER SUPPORT SCORES (append at end of response):
Rate how strongly the evidence supports each tier, 0-100 independently.
  0   = no evidence supports this tier
  50  = ambiguous
  100 = evidence overwhelmingly supports this tier

SCORE_BASELINE_T2D: [0-100]
SCORE_MILD_COMPLICATIONS: [0-100]
SCORE_MODERATE_COMPLICATIONS: [0-100]
SCORE_ADVANCED_CRITICAL: [0-100]
"""

        task_assess_gpt = Task(
            description=ASSESS_PROMPT,
            expected_output=(
                "Structured T2D severity assessment with domain-by-domain evidence, "
                "beyond-framework clinical observations, four-tier and binary "
                "classification, confidence level, and four tier support scores."
            ),
            agent=assessor_gpt,
            async_execution=True
        )

        task_assess_deepseek = Task(
            description=ASSESS_PROMPT,
            expected_output=(
                "Structured T2D severity assessment with domain-by-domain evidence, "
                "beyond-framework clinical observations, four-tier and binary "
                "classification, confidence level, and four tier support scores."
            ),
            agent=assessor_deepseek,
            async_execution=True
        )

        task_consolidate = Task(
            description=f"""
You have received T2D severity assessments from two independent clinical assessors
for the same patient at their T5 reclassification point.
Patient ID: {patient_id}

REMINDER: All patients have confirmed T2D. Sparse record = Baseline T2D.
Absence of complications IS a finding. Never use Insufficient Data.

=================================================================
CRITICAL INSTRUCTION: OUTPUT THE ===FINAL=== BLOCK FIRST.
Write it as the VERY FIRST THING in your response, before any analysis.
This is mandatory — the block must appear at the start of your output.
=================================================================

===FINAL===
PATIENT_ID: {patient_id}
FOUR_TIER: [Baseline T2D / Mild Complications / Moderate Complications / Advanced/Critical]
BINARY: [COMPLEX / NOT_COMPLEX]
ASSESSOR_A_TIER: [Dr. A four-tier]
ASSESSOR_B_TIER: [Dr. B four-tier]
ASSESSOR_A_BINARY: [Dr. A binary]
ASSESSOR_B_BINARY: [Dr. B binary]
CONFIDENCE: [High / Medium / Low]
AGREEMENT: [Full / Partial / None]
INDEX_DATE_CONTEXT: {INDEX_DATE_LABEL}
KEY_EVIDENCE: [One sentence — single most decisive piece of evidence]
SCORE_BASELINE_T2D: [0-100]
SCORE_MILD_COMPLICATIONS: [0-100]
SCORE_MODERATE_COMPLICATIONS: [0-100]
SCORE_ADVANCED_CRITICAL: [0-100]
===END===

After the block, briefly provide:
- Where assessors agreed / disagreed and your resolution
- Key evidence driving the classification
- Tier score rationale (each score 0-100 independently)
- Any Synthea data quality notes
""",
            expected_output=(
                "===FINAL=== block FIRST (immediately, before any analysis), "
                "then brief agreement analysis and evidence summary."
            ),
            agent=consolidator,
            context=[task_assess_gpt, task_assess_deepseek]
        )

        # ── Run crew ──────────────────────────────────────────
        crew = Crew(
            agents=[assessor_gpt, assessor_deepseek, consolidator],
            tasks=[task_assess_gpt, task_assess_deepseek, task_consolidate],
            verbose=True
        )

        patient_start = time.time()

        try:
            crew_result     = crew.kickoff()
            patient_elapsed = time.time() - patient_start

            # Save assessor outputs
            save_and_track("openai",   f"assessment_{patient_short}", str(task_assess_gpt.output))
            save_and_track("deepseek", f"assessment_{patient_short}", str(task_assess_deepseek.output))
            # NOTE: consolidated is saved after result_text is resolved (full, untruncated)

            completed_patients.add(patient_id)
            patients_since_checkpoint += 1

            # ── Robust parser for ===FINAL=== block ──────────
            # str(crew_result) truncates in many CrewAI versions — use .raw instead
            if hasattr(crew_result, 'raw') and crew_result.raw:
                result_text = crew_result.raw
            elif hasattr(crew_result, 'result') and crew_result.result:
                result_text = crew_result.result
            elif hasattr(crew_result, 'output') and crew_result.output:
                result_text = crew_result.output
            else:
                result_text = str(crew_result)

            # Backup: get the consolidator task output directly —
            # most reliable source since it's the raw Claude response
            consolidator_raw = ""
            if hasattr(task_consolidate, 'output') and task_consolidate.output:
                if hasattr(task_consolidate.output, 'raw'):
                    consolidator_raw = task_consolidate.output.raw or ""
                elif hasattr(task_consolidate.output, 'result'):
                    consolidator_raw = task_consolidate.output.result or ""
                else:
                    consolidator_raw = str(task_consolidate.output)

            # Use whichever is longer or actually contains the FINAL block
            if "===FINAL===" not in result_text and "===FINAL===" in consolidator_raw:
                print(f"  Using task.output directly (crew_result was truncated)")
                result_text = consolidator_raw
            elif len(consolidator_raw) > len(result_text):
                result_text = consolidator_raw

            print(f"  Result: {len(result_text)} chars | "
                  f"===FINAL===: {'YES' if '===FINAL===' in result_text else 'NO'}")

            # Also fix the save to use full text (no truncation)
            save_and_track("consolidated", f"classification_{patient_short}", result_text)

            def parse_final_block(text):
                """
                Robustly extract the ===FINAL=== block and parse key:value pairs.
                Handles: missing ===END===, brackets in values, spaces around colons,
                multi-line fields, duplicate blocks (takes the LAST one), and
                CrewAI wrapper text around the actual output.
                """
                import re

                # Take the LAST occurrence of ===FINAL=== in case of duplicates
                # (e.g. Claude echoes the format instructions before filling them in)
                parts = text.split("===FINAL===")
                if len(parts) < 2:
                    return {}, "NO_FINAL_BLOCK"

                block = parts[-1]  # last occurrence

                # Strip ===END=== if present; if not, use everything after ===FINAL===
                if "===END===" in block:
                    block = block.split("===END===")[0]
                else:
                    # Take only the first ~30 lines after ===FINAL=== as a safety cap
                    block = "\n".join(block.strip().split("\n")[:30])

                result = {}
                for line in block.strip().split("\n"):
                    line = line.strip()
                    if not line or line.startswith("#"):
                        continue
                    # Match KEY: value  (allow spaces around colon)
                    m = re.match(r'^([A-Z][A-Z0-9_]+)\s*:\s*(.*)$', line)
                    if not m:
                        continue
                    key = m.group(1).strip()
                    val = m.group(2).strip()
                    # Strip surrounding brackets, quotes, backticks
                    val = re.sub(r'^[\[\(\`"\']|[\]\)\`"\']$', '', val).strip()
                    # Skip if value looks like a placeholder e.g. "Dr. A four-tier"
                    if val and not val.startswith("Dr.") and not val.startswith("["):
                        result[key] = val

                return result, "OK"

            parsed, parse_status = parse_final_block(result_text)

            if parse_status != "OK":
                print(f"  WARNING: {parse_status} for {patient_short} — "
                      f"attempting free-text fallback")

            # ── Tier extraction with fuzzy fallback ───────────
            def extract_tier(raw_val):
                """
                Map raw string to a valid tier label.
                Handles: brackets, partial matches, case variations.
                """
                if not raw_val:
                    return "PARSE_ERROR"
                v = raw_val.strip().strip("[]()").lower()
                # Exact match first (case-insensitive)
                for tier in VALID_TIERS:
                    if v == tier.lower():
                        return tier
                # Partial match — longest match wins
                matches = [t for t in VALID_TIERS if t.lower() in v or v in t.lower()]
                if len(matches) == 1:
                    return matches[0]
                # Keyword fallback
                if "advanced" in v or "critical" in v:
                    return "Advanced/Critical"
                if "moderate" in v:
                    return "Moderate Complications"
                if "mild" in v and "baseline" not in v:
                    return "Mild Complications"
                if "baseline" in v or "no complication" in v:
                    return "Baseline T2D"
                # Last resort: scan full result text for tier mentions near FOUR_TIER
                for tier in VALID_TIERS:
                    pattern = f"FOUR_TIER.*{re.escape(tier)}"
                    if re.search(pattern, result_text, re.IGNORECASE):
                        return tier
                return "PARSE_ERROR"

            import re
            four_tier = extract_tier(parsed.get("FOUR_TIER", ""))

            # ── Full-text fallback when ===FINAL=== block is missing ──
            # Claude sometimes writes the verdict in prose/table form instead.
            # Patterns we've seen in real outputs:
            #   **Final Tier** | **Moderate Complications**
            #   Final Classification: Moderate Complications
            #   FOUR_TIER: Moderate Complications  (outside the block)
            #   I classify this patient as Moderate Complications
            if four_tier == "PARSE_ERROR" and parse_status == "NO_FINAL_BLOCK":
                print(f"  Attempting full-text tier extraction...")

                def fulltext_extract_tier(text):
                    # Pattern 1: markdown table row  **Final Tier** | **X**
                    m = re.search(
                        r'\*\*Final Tier\*\*\s*\|[^|]*\|\s*\*\*([^*]+)\*\*',
                        text, re.IGNORECASE)
                    if m:
                        return extract_tier(m.group(1))

                    # Pattern 2: Final Tier** | **X** (single pipe)
                    m = re.search(
                        r'Final Tier[*\s|]+\*\*([^*\n]+)\*\*',
                        text, re.IGNORECASE)
                    if m:
                        return extract_tier(m.group(1))

                    # Pattern 3: "Final Classification: X" or "Final Tier: X"
                    m = re.search(
                        r'(?:Final (?:Tier|Classification)|FOUR_TIER)\s*[:\|]\s*\**([^\n*|]+)',
                        text, re.IGNORECASE)
                    if m:
                        return extract_tier(m.group(1).strip())

                    # Pattern 4: "classify this patient as X" / "classified as X"
                    m = re.search(
                        r'classif(?:y|ied)[^.]*?\bas\b\s+([A-Z][^\n.,]+)',
                        text, re.IGNORECASE)
                    if m:
                        t = extract_tier(m.group(1).strip())
                        if t != "PARSE_ERROR":
                            return t

                    # Pattern 5: last occurrence of a tier name near the end of text
                    # (take last 3000 chars — where the verdict usually is)
                    tail = text[-3000:]
                    last_pos = -1
                    last_tier = None
                    for tier in VALID_TIERS:
                        pos = tail.rfind(tier)
                        if pos > last_pos:
                            last_pos = pos
                            last_tier = tier
                    if last_tier:
                        return last_tier

                    return "PARSE_ERROR"

                four_tier = fulltext_extract_tier(result_text)
                if four_tier != "PARSE_ERROR":
                    print(f"  Full-text fallback succeeded: {four_tier}")
                else:
                    print(f"  Full-text fallback also failed — needs manual review")

            # Warn if still couldn't parse
            if four_tier == "PARSE_ERROR":
                print(f"  WARNING: could not parse FOUR_TIER for {patient_short}")
                print(f"  Raw parsed dict: {parsed}")
                # Save raw output for manual review
                save_and_track("parse_errors",
                               f"raw_output_{patient_short}",
                               result_text)  # full text, no truncation

            # ── Binary: derive from tier if not cleanly parsed ─
            binary_raw = parsed.get("BINARY", "")
            if "complex" in binary_raw.lower() and "not" not in binary_raw.lower():
                binary = "COMPLEX"
            elif "not_complex" in binary_raw.lower() or "not complex" in binary_raw.lower():
                binary = "NOT_COMPLEX"
            elif four_tier in VALID_TIERS:
                # Always derive from tier as ground truth
                binary = "COMPLEX" if four_tier in COMPLEX_TIERS else "NOT_COMPLEX"
            else:
                binary = "PARSE_ERROR"

            # Parse scores
            s_base = parse_score(parsed, "SCORE_BASELINE_T2D")
            s_mild = parse_score(parsed, "SCORE_MILD_COMPLICATIONS")
            s_mod  = parse_score(parsed, "SCORE_MODERATE_COMPLICATIONS")
            s_adv  = parse_score(parsed, "SCORE_ADVANCED_CRITICAL")
            softmax = softmax_scores(s_base, s_mild, s_mod, s_adv)

            patient_result = {
                "patient_id"                  : patient_id,
                "patient_short"               : patient_short,
                "index_date"                  : INDEX_DATE_LABEL,
                "status"                      : "SUCCESS",
                "time_seconds"                : round(patient_elapsed, 1),
                "four_tier"                   : four_tier,
                "binary"                      : binary,
                "agreement"                   : parsed.get("AGREEMENT",       "?"),
                "confidence"                  : parsed.get("CONFIDENCE",      "?"),
                "assessor_a_tier"             : parsed.get("ASSESSOR_A_TIER", "?"),
                "assessor_b_tier"             : parsed.get("ASSESSOR_B_TIER", "?"),
                "key_evidence"                : parsed.get("KEY_EVIDENCE",    "?"),
                "score_baseline_t2d"          : s_base,
                "score_mild_complications"    : s_mild,
                "score_moderate_complications": s_mod,
                "score_advanced_critical"     : s_adv,
                **softmax,
            }
            all_results.append(patient_result)

            scores_str = (f"BT:{s_base} MC:{s_mild} "
                          f"MoC:{s_mod} AC:{s_adv}")
            print(f"\n  Patient {patient_short} — {four_tier} / {binary} | "
                  f"Agreement: {parsed.get('AGREEMENT','?')} | "
                  f"Conf: {parsed.get('CONFIDENCE','?')} | "
                  f"Scores: [{scores_str}] | "
                  f"{patient_elapsed:.0f}s")

        except Exception as e:
            patient_elapsed = time.time() - patient_start
            error_msg       = str(e)[:200]
            print(f"\n  Patient {patient_short} FAILED after {patient_elapsed:.0f}s: {error_msg}")

            errors.append({
                "patient_id"   : patient_id,
                "patient_short": patient_short,
                "index_date"   : INDEX_DATE_LABEL,
                "error"        : error_msg,
                "time_seconds" : round(patient_elapsed, 1),
            })
            all_results.append({
                "patient_id"                  : patient_id,
                "patient_short"               : patient_short,
                "index_date"                  : INDEX_DATE_LABEL,
                "status"                      : f"FAILED: {error_msg}",
                "time_seconds"                : round(patient_elapsed, 1),
                "four_tier"                   : "ERROR",
                "binary"                      : "ERROR",
                "agreement"                   : "ERROR",
                "confidence"                  : "ERROR",
                "assessor_a_tier"             : "ERROR",
                "assessor_b_tier"             : "ERROR",
                "key_evidence"                : "ERROR",
                "score_baseline_t2d"          : None,
                "score_mild_complications"    : None,
                "score_moderate_complications": None,
                "score_advanced_critical"     : None,
                "prob_baseline_t2d"           : None,
                "prob_mild_complications"     : None,
                "prob_moderate_complications" : None,
                "prob_advanced_critical"      : None,
            })
            patients_since_checkpoint += 1

        # ── Periodic checkpoint ───────────────────────────────
        if patients_since_checkpoint >= CHECKPOINT_EVERY:
            print(f"\n  Auto-checkpoint: {len(completed_patients)} patients done...")
            save_and_track("metadata", "pipeline_results_partial", all_results)
            if errors:
                save_and_track("metadata", "pipeline_errors", errors)
            with open(f"{INDEX_SUBDIR}/checkpoint.json", "w") as f:
                json.dump({
                    "completed_patients": list(completed_patients),
                    "total_patients"    : len(ALL_PATIENT_IDS),
                    "errors"            : len(errors),
                    "timestamp"         : time.strftime("%Y-%m-%d %H:%M:%S"),
                }, f, indent=2)
            try:
                download_checkpoint_zip(completed_patients, all_results, errors)
            except Exception as ce:
                print(f"  Checkpoint download failed: {ce}")
            patients_since_checkpoint = 0

    # ── Pipeline complete ─────────────────────────────────────
    pipeline_elapsed = time.time() - pipeline_start
    save_and_track("metadata", "pipeline_results_final", all_results)
    if errors:
        save_and_track("metadata", "pipeline_errors_final", errors)

    print(f"\n{'=' * 65}")
    print(f"PIPELINE COMPLETE — {INDEX_DATE_DISPLAY}")
    print(f"{'=' * 65}")
    print(f"  Total time      : {pipeline_elapsed/60:.1f} min")
    print(f"  Completed       : {len(completed_patients)} / {len(ALL_PATIENT_IDS)}")
    print(f"  Errors          : {len(errors)}")

    success = [r for r in all_results if r["status"] == "SUCCESS"]
    if success:
        from collections import Counter
        tier_counts      = Counter(r["four_tier"] for r in success)
        binary_counts    = Counter(r["binary"]    for r in success)
        agreement_counts = Counter(r["agreement"] for r in success)
        times            = [r["time_seconds"] for r in success]

        print(f"  Avg time/patient: {sum(times)/len(times):.0f}s")
        print(f"\n  Four-Tier Distribution:")
        for tier in VALID_TIERS:
            count = tier_counts.get(tier, 0)
            print(f"    {tier:<30}: {count:>3}  {'█' * count}")
        print(f"\n  Binary Distribution:")
        for label, count in sorted(binary_counts.items()):
            print(f"    {label:<15}: {count:>3}")
        print(f"\n  Inter-Assessor Agreement:")
        for level, count in sorted(agreement_counts.items()):
            print(f"    {level:<10}: {count:>3}")

    if errors:
        print(f"\n  Failed patients:")
        for e in errors:
            print(f"    {e['patient_short']}: {e['error'][:80]}")

    remaining_after = [p for p in ALL_PATIENT_IDS if p not in completed_patients]
    if remaining_after:
        print(f"\n  {len(remaining_after)} patients remaining.")
        print(f"  Re-run this cell to process the next batch of {BATCH_SIZE}.")
    else:
        print(f"\n  ALL {len(ALL_PATIENT_IDS)} PATIENTS COMPLETE for {INDEX_DATE_LABEL}!")

    # Final checkpoint
    try:
        download_checkpoint_zip(completed_patients, all_results, errors)
    except Exception as e:
        print(f"  Final checkpoint failed: {e}")


# ── Download everything when all 200 patients done ────────────
# Uncomment and run after all patients are complete:

# zip_name = f"/content/phase2_{INDEX_DATE_LABEL}_FINAL.zip"
# with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
#     for root, dirs, files_list in os.walk(INDEX_SUBDIR):
#         for fname in files_list:
#             fpath   = os.path.join(root, fname)
#             arcname = os.path.relpath(fpath, OUTPUT_DIR)
#             zf.write(fpath, arcname)
# size_mb = os.path.getsize(zip_name) / (1024 * 1024)
# print(f"FINAL ZIP: {zip_name} — {size_mb:.1f} MB")
# colab_files.download(zip_name)

In [ ]:
# ══════════════════════════════════════════════════════════════
# POST-BATCH AUDIT — Run after each batch completes
# ══════════════════════════════════════════════════════════════
import os, json
from collections import Counter

INDEX_SUBDIR = "/content/pipeline_outputs/phase2/T5_reclassification"

# ── 1. Summary of current results ────────────────────────────
print("=" * 65)
print("POST-BATCH AUDIT")
print("=" * 65)

success   = [r for r in all_results if r["status"] == "SUCCESS"]
errors    = [r for r in all_results if r["status"].startswith("FAILED")]
parse_err = [r for r in success if r["four_tier"] == "PARSE_ERROR"]
good      = [r for r in success if r["four_tier"] != "PARSE_ERROR"]

print(f"\n  Total processed this session : {len(all_results)}")
print(f"  Crew ran successfully        : {len(success)}")
print(f"  Crew crashed (FAILED)        : {len(errors)}")
print(f"  Crew OK but parse failed     : {len(parse_err)}")
print(f"  Clean results                : {len(good)}")

# ── 2. Tier distribution for clean results ────────────────────
if good:
    print(f"\n  Tier distribution (clean only):")
    for tier, n in Counter(r["four_tier"] for r in good).most_common():
        print(f"    {tier:<30} {n:>3}  {'█' * n}")

# ── 3. Parse error patients — show raw output snippet ─────────
if parse_err:
    print(f"\n  PARSE ERROR PATIENTS ({len(parse_err)}):")
    print(f"  {'Short ID':<12} {'Status':<10} {'Time':>6}s")
    print(f"  {'-'*35}")
    for r in parse_err:
        print(f"  {r['patient_short']:<12} parse_err  {r['time_seconds']:>6.0f}s")

    print(f"\n  Checking raw saved outputs...")
    for r in parse_err:
        short = r['patient_short']
        # Check all possible save locations
        paths = [
            f"{INDEX_SUBDIR}/parse_errors/raw_output_{short}.txt",
            f"{INDEX_SUBDIR}/consolidated/classification_{short}.txt",
        ]
        for path in paths:
            if os.path.exists(path):
                with open(path) as f:
                    content = f.read()
                print(f"\n  {'─'*65}")
                print(f"  Patient: {short}  ({len(content)} chars)")
                print(f"  File: {os.path.basename(path)}")
                # Show last 1500 chars — where ===FINAL=== should be
                print(f"  --- LAST 1500 CHARS (where ===FINAL=== should appear) ---")
                print(content[-1500:])
                print(f"  --- SEARCH: does '===FINAL===' appear anywhere? ---")
                print(f"  {'YES' if '===FINAL===' in content else 'NO — block is missing entirely'}")
                if '===FINAL===' in content:
                    idx = content.rfind('===FINAL===')
                    print(f"  Found at position {idx} of {len(content)}")
                    print(f"  Text around it: ...{content[idx:idx+200]}...")
                break
        else:
            print(f"\n  {short}: no saved output found (crew probably crashed silently)")

# ── 4. FAILED patients (crew crashed) ────────────────────────
if errors:
    print(f"\n  CRASHED PATIENTS ({len(errors)}):")
    for r in errors:
        print(f"    {r['patient_short']}: {r['status'][:100]}")

# ── 5. Patients to re-run ─────────────────────────────────────
needs_rerun = [r["patient_id"] for r in parse_err + errors]
print(f"\n{'=' * 65}")
print(f"  Patients needing re-run: {len(needs_rerun)}")
if needs_rerun:
    print(f"  IDs:")
    for pid in needs_rerun:
        print(f"    {pid}")
    print(f"\n  To re-run these, paste into Cell B before running:")
    print(f"  completed_patients -= set({[p for p in needs_rerun]})")